# LinkedIn IT Job Scraper (Sri Lanka) - Keyword-by-Keyword Strategy

## Project: Skill-Aware Job Matching for Sri Lankan IT Professionals

This notebook scrapes **IT-specific** job postings from LinkedIn in Sri Lanka using a **keyword-by-keyword approach** for maximum coverage.

### New Scraping Strategy:
1. Individual Keyword Search: Each IT keyword is searched separately (100+ keywords)
2. Complete Pagination: Scrapes ALL available pages for each keyword
3. Progressive Saving: After each keyword, results are saved/appended to CSV
4. No Filter Combinations: Simplified approach - just keyword + location
5. Maximum Coverage: Gets every possible IT job by searching all relevant terms

### Search Keywords Include:
- Roles: developer, engineer, analyst, designer, manager, architect, etc.
- Specializations: frontend, backend, fullstack, devops, data scientist, QA, etc.
- Technologies: python, java, aws, azure, react, kubernetes, etc.
- Domains: software, IT, technology, security, cloud, data, AI, etc.

### Data Fields Extracted:
- Job ID, Title, Company Name, Location, Posted Date
- Job Description (full text)
- Experience Level, Employment Type, Job Function, Industries
- Required Skills (500+ IT skills extracted)
- Job Criteria, Number of Applicants, Job URL
- Search Keyword (for tracking which keyword found the job)
- Scraped Timestamp

### Why Keyword-by-Keyword?
- Maximum Coverage: Each keyword gets separate searches
- No Missed Jobs: Different keywords surface different jobs
- Progressive Saving: Data saved after each keyword (safe from failures)
- Resumable: Can stop and resume from last keyword
- Better Deduplication: Tracks which keywords found which jobs

### Output:
- Single CSV file that grows with each keyword
- Excel file (final dataset)
- Summary report with keyword statistics

In [1]:
# Step 1: Install Required Packages
!pip install requests beautifulsoup4 pandas tqdm openpyxl lxml

In [2]:
import requests
from bs4 import BeautifulSoup
import math
import pandas as pd
from datetime import datetime
from pathlib import Path
from tqdm import tqdm
import logging
import time
import random
import re

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler()]
)
logger = logging.getLogger(__name__)

# Multiple user agents to rotate
USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.0 Safari/605.1.15",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:122.0) Gecko/20100101 Firefox/122.0",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
]

# Configuration - KEYWORD-BY-KEYWORD SCRAPING
CONFIG = {
    'location': 'Sri Lanka',
    'output_dir': 'data',
    'output_filename': 'linkedin_sri_lanka_IT_jobs_progressive.csv',
    'request_delay': (2, 4),  # Delay between requests (seconds)
    'page_delay': (3, 6),  # Delay between pages (seconds)
    'keyword_delay': (10, 15),  # Delay between keywords (seconds)
    'max_pages_per_keyword': 40,  # Max pages to scrape per keyword (LinkedIn typically shows ~1000 results max = 40 pages)

    # Comprehensive IT job search keywords - Each searched separately
    'search_keywords': [

        # ==============================
        # --- Core Software Roles ---
        # ==============================
        'software engineer', 'software developer', 'software development engineer',
        'junior software engineer', 'senior software engineer', 'principal software engineer',
        'staff software engineer', 'lead software engineer',
        'web developer', 'full stack developer', 'backend developer', 'frontend developer',
        'mobile developer', 'application developer', 'enterprise application developer',
        'game developer', 'desktop application developer',

        # ==============================
        # --- Engineering Leadership ---
        # ==============================
        'technical lead', 'team lead', 'engineering manager',
        'senior engineering manager', 'director of engineering',
        'vp engineering', 'cto', 'cio', 'head of engineering',
        'chief technology officer', 'chief information officer',

        # ==============================
        # --- DevOps / Cloud / Infra ---
        # ==============================
        'devops engineer', 'site reliability engineer', 'sre',
        'cloud engineer', 'cloud architect', 'cloud consultant',
        'platform engineer', 'infrastructure engineer',
        'build engineer', 'release engineer', 'configuration engineer',
        'aws engineer', 'aws architect',
        'azure engineer', 'azure architect',
        'gcp engineer', 'gcp architect',
        'cloud security engineer', 'cloud operations engineer',
        'kubernetes engineer', 'docker engineer',
        'terraform engineer', 'linux engineer',

        # ==============================
        # --- Data & AI ---
        # ==============================
        'data scientist', 'senior data scientist',
        'data analyst', 'business intelligence analyst',
        'data engineer', 'big data engineer',
        'analytics engineer', 'bi developer',
        'etl developer', 'data architect',
        'machine learning engineer', 'ml engineer',
        'mlops engineer', 'ai engineer',
        'ai researcher', 'deep learning engineer',
        'nlp engineer', 'computer vision engineer',
        'generative ai engineer', 'llm engineer',
        'prompt engineer', 'ai solutions architect',
        'research scientist ai',

        # ==============================
        # --- Cybersecurity ---
        # ==============================
        'security engineer', 'security analyst',
        'cybersecurity specialist', 'cyber security engineer',
        'penetration tester', 'ethical hacker',
        'incident responder', 'soc analyst',
        'security operations engineer',
        'application security engineer',
        'cloud security architect',
        'information security analyst',
        'governance risk compliance', 'grc analyst',

        # ==============================
        # --- QA / Testing ---
        # ==============================
        'qa engineer', 'quality assurance engineer',
        'test engineer', 'automation engineer',
        'manual tester', 'performance tester',
        'qa analyst', 'sdet', 'software test engineer',

        # ==============================
        # --- Product / Project ---
        # ==============================
        'product manager', 'technical product manager',
        'project manager', 'it project manager',
        'program manager', 'scrum master',
        'agile coach', 'delivery manager',
        'release manager', 'business analyst',
        'technical business analyst',
        'product owner',

        # ==============================
        # --- UI / UX / Design ---
        # ==============================
        'ui designer', 'ux designer',
        'ui ux designer', 'product designer',
        'interaction designer', 'visual designer',
        'ux researcher', 'design system engineer',

        # ==============================
        # --- Technology-Specific ---
        # ==============================
        'python developer', 'django developer', 'fastapi developer',
        'java developer', 'spring boot developer',
        'javascript developer', 'typescript developer',
        'react developer', 'angular developer', 'vue developer',
        'node.js developer', 'express developer',
        'php developer', 'laravel developer',
        '.net developer', 'c# developer', 'asp.net developer',
        'golang developer', 'ruby developer',
        'ruby on rails developer',
        'flutter developer', 'react native developer',
        'ios developer', 'android developer',
        'kotlin developer', 'swift developer',
        'salesforce developer', 'salesforce consultant',
        'sap consultant', 'sap abap developer',
        'oracle developer', 'oracle dba',
        'wordpress developer', 'shopify developer',
        'magento developer',

        # ==============================
        # --- Embedded / Hardware ---
        # ==============================
        'embedded systems engineer', 'firmware engineer',
        'hardware engineer', 'iot engineer',
        'robotics engineer', 'fpga engineer',
        'electronics engineer',

        # ==============================
        # --- Blockchain / Web3 ---
        # ==============================
        'blockchain developer', 'web3 developer',
        'smart contract developer', 'solidity developer',
        'crypto engineer',

        # ==============================
        # --- Database ---
        # ==============================
        'database administrator', 'dba',
        'mysql dba', 'postgresql dba',
        'mongodb engineer', 'database engineer',

        # ==============================
        # --- IT Support / Operations ---
        # ==============================
        'it specialist', 'it support engineer',
        'technical support engineer',
        'help desk technician', 'service desk analyst',
        'system administrator', 'system admin',
        'network engineer', 'network administrator',
        'it operations engineer',
        'desktop support engineer',

        # ==============================
        # --- Architecture ---
        # ==============================
        'software architect', 'solutions architect',
        'enterprise architect', 'technical architect',
        'application architect', 'data architect',

        # ==============================
        # --- Emerging & Niche ---
        # ==============================
        'quantum computing engineer',
        'ar developer', 'vr developer',
        'ar vr developer', 'mixed reality developer',
        'metaverse developer',
        'digital transformation consultant',
        'automation architect',
        'rpa developer', 'ui path developer',
        'low code developer', 'no code developer',

        # ==============================
        # --- Contract / Freelance Variants ---
        # ==============================
        'contract software engineer',
        'freelance developer',
        'remote software engineer',
        'part time developer',
        'intern software engineer',
        'graduate software engineer',
    ],
}

def get_random_headers():
    """Get random headers to avoid detection"""
    return {
        "User-Agent": random.choice(USER_AGENTS),
        "Accept-Language": "en-US,en;q=0.9",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Referer": "https://www.linkedin.com",
        "Connection": "keep-alive",
    }

logger.info("Configuration loaded successfully")
logger.info(f"Total search keywords: {len(CONFIG['search_keywords'])}")

In [3]:
# Step 3: Define HTTP Request Helper
def safe_get(url, max_retries=3):
    """Make HTTP GET request with retry logic"""
    for attempt in range(max_retries):
        try:
            response = requests.get(url, headers=get_random_headers(), timeout=10)

            if response.status_code == 429:
                logger.warning(f"Rate limited. Waiting... (Attempt {attempt + 1}/{max_retries})")
                time.sleep(30 * (attempt + 1))
                continue

            if response.status_code == 200:
                return response

            logger.warning(f"Status code {response.status_code} on attempt {attempt + 1}")

        except Exception as e:
            logger.error(f"Request error on attempt {attempt + 1}: {e}")
            if attempt < max_retries - 1:
                time.sleep(5 * (attempt + 1))

    return None

# Test the function
test_url = "https://www.linkedin.com/jobs-guest/jobs/api/seeMoreJobPostings/search?location=Sri%20Lanka&start=0"
response = safe_get(test_url)
if response:
    print(f"HTTP request function working! Status: {response.status_code}")
else:
    print("HTTP request function failed. Check your connection.")

HTTP request function working! Status: 200


In [4]:
def extract_skills_from_text(text):
    """
    Extract IT skills from job description text
    Optimized: Uses a single compiled regex with caching and improved boundary detection
    """
    if not text:
        return []

    # Use function attribute for caching to avoid re-compiling 2000+ regexes
    if not hasattr(extract_skills_from_text, 'pattern'):
        # Comprehensive IT skills dictionary - World's Largest IT Skill Taxonomy
        SKILLS = [
            # --- Programming Languages ---
            'python', 'java', 'javascript', 'typescript', 'c++', 'c#',
            'c', 'php', 'ruby', 'go', 'golang', 'rust',
            'kotlin', 'swift', 'scala', 'r', 'perl', 'matlab',
            'julia', 'dart', 'objective-c', 'vb.net', 'visual basic', 'visual basic .net',
            'groovy', 'lua', 'shell', 'bash', 'powershell', 'assembly',
            'clojure', 'elixir', 'haskell', 'f#', 'erlang', 'ocaml',
            'lisp', 'prolog', 'scheme', 'fortran', 'cobol', 'pascal',
            'delphi', 'ada', 'abap', 'apex', 'sas', 'pl/sql',
            't-sql', 'vba', 'scratch', 'logo', 'wolfram', 'mathematica',
            'labview', 'opencl', 'cuda', 'verilog', 'vhdl', 'zig',
            'carbon', 'nim', 'crystal', 'd', 'solidity', 'vyper',
            'yul', 'move', 'cairo', 'elm', 'purescript', 'reasonml',
            'rescript', 'idris', 'agda', 'coq', 'lean', 'v',
            'odin', 'haxe', 'red', 'rebol', 'tcl', 'awk',
            'sed', 'smalltalk', 'racket', 'alice',

            # --- Web Frontend ---
            'html', 'html5', 'xhtml', 'dhtml', 'css', 'css3',
            'sass', 'scss', 'less', 'stylus', 'xml', 'json',
            'yaml', 'toml', 'markdown', 'svg', 'canvas', 'webgl',
            'webrtc', 'websocket', 'react', 'react.js', 'reactjs', 'angular',
            'angular.js', 'angularjs', 'vue', 'vue.js', 'vuejs', 'svelte',
            'sveltekit', 'solidjs', 'solid.js', 'preact', 'alpine.js', 'alpinejs',
            'lit', 'lit-element', 'stencil', 'stenciljs', 'ember', 'ember.js',
            'backbone', 'backbone.js', 'knockout', 'knockout.js', 'jquery', 'zepto',
            'mootools', 'dojo', 'extjs', 'sencha touch', 'aurelia', 'mithril',
            'riot.js', 'next.js', 'nextjs', 'nuxt', 'nuxt.js', 'remix',
            'remix.run', 'gatsby', 'astro', 'qwik', 'fresh', 'blitz.js',
            'redwoodjs', 'meteor', 'mean stack', 'mern stack', 'mevn stack', 'lamp stack',
            'tailwind', 'tailwind css', 'bootstrap', 'material ui', 'mui', 'chakra ui',
            'ant design', 'antd', 'semantic ui', 'foundation', 'bulma', 'picocss',
            'windi css', 'unocss', 'vanilla extract', 'emotion', 'styled-components', 'jss',
            'radix ui', 'headless ui', 'shadcn', 'daisyui', 'mantine', 'primevue',
            'primereact', 'primeng', 'vuetify', 'quasar', 'buefy', 'webpack',
            'vite', 'rollup', 'parcel', 'esbuild', 'swc', 'turbopack',
            'browserify', 'gulp', 'grunt', 'babel', 'tsc', 'yeoman',
            'bower', 'npm', 'yarn', 'pnpm', 'bun', 'lerna',
            'nx', 'turborepo',

            # --- Web Backend ---
            'node.js', 'nodejs', 'deno', 'express', 'express.js', 'fastify',
            'koa', 'hapi', 'loopback', 'nest.js', 'nestjs', 'adonisjs',
            'sails.js', 'meteor.js', 'strapi', 'keystone', 'django', 'flask',
            'fastapi', 'pyramid', 'tornado', 'bottle', 'sanic', 'falcon',
            'cherrypy', 'celery', 'sqlalchemy', 'peewee', 'pydantic', 'gunicorn',
            'uwsgi', 'spring', 'spring boot', 'spring mvc', 'spring data', 'spring security',
            'spring cloud', 'jakarta ee', 'j2ee', 'java ee', 'hibernate', 'jpa',
            'mybatis', 'struts', 'jsf', 'vaadin', 'wicket', 'play framework',
            'grails', 'quarkus', 'micronaut', 'helidon', 'vert.x', 'akka',
            'laravel', 'symfony', 'codeigniter', 'cakephp', 'yii', 'zend framework',
            'laminas', 'slim', 'phalcon', 'fuelphp', 'phpspec', 'phpunit',
            'composer', 'wordpress', 'drupal', 'joomla', 'magento', '.net',
            '.net core', 'asp.net', 'asp.net core', 'ado.net', 'entity framework', 'dapper',
            'blazor', 'razor', 'maui', 'wcf', 'wpf', 'winforms',
            'uwp', 'silverlight', 'ruby on rails', 'rails', 'sinatra', 'hanami',
            'padrino', 'active record', 'rspec', 'capybara', 'gin', 'echo',
            'fiber', 'beego', 'revel', 'buffalo', 'chi', 'gorilla',
            'gorm',

            # --- Mobile Development ---
            'react native', 'flutter', 'xamarin', 'xamarin.forms', 'dotnet maui', 'ionic',
            'cordova', 'phonegap', 'capacitor', 'nativescript', 'quasar framework', 'unity mobile',
            'unreal mobile', 'android', 'android sdk', 'android studio', 'kotlin android', 'java android',
            'jetpack compose', 'retrofit', 'dagger', 'hilt', 'room', 'livedata',
            'glide', 'picasso', 'espresso', 'robolectric', 'ios', 'ios sdk',
            'xcode', 'swift', 'swiftui', 'uikit', 'cocoa', 'cocoa touch',
            'objective-c', 'core data', 'alamofire', 'kingfisher', 'snapkit', 'rxswift',
            'combine', 'testflight', 'app store connect',

            # --- Game Development & Graphics ---
            'unity', 'unity3d', 'unreal engine', 'unreal', 'ue4', 'ue5',
            'godot', 'cryengine', 'gamemaker', 'construct 3', 'id tech', 'source engine',
            'rpg maker', 'renpy', 'cocos2d', 'libgdx', 'monogame', 'opengl',
            'webgl', 'vulkan', 'directx', 'metal', 'hlsl', 'glsl',
            'shader', 'three.js', 'babylon.js', 'pixijs', 'phaser', 'a-frame',
            'blender', 'maya', '3ds max', 'zbrush', 'houdini', 'cinema 4d',

            # --- Data Science, AI & ML ---
            'data science', 'machine learning', 'ml', 'deep learning', 'dl', 'artificial intelligence',
            'ai', 'statistics', 'linear algebra', 'calculus', 'probability', 'numpy',
            'pandas', 'scipy', 'scikit-learn', 'sklearn', 'matplotlib', 'seaborn',
            'plotly', 'bokeh', 'statsmodels', 'sympy', 'polars', 'dask',
            'vaex', 'modin', 'tensorflow', 'tf', 'pytorch', 'keras',
            'caffe', 'theano', 'mxnet', 'cntk', 'chainer', 'paddlepaddle',
            'jax', 'flax', 'haiku', 'trax', 'fastai', 'lightning',
            'pytorch lightning', 'nlp', 'natural language processing', 'nltk', 'spacy', 'gensim',
            'textblob', 'corenlp', 'transformers', 'hugging face', 'bert', 'gpt',
            'gpt-3', 'gpt-4', 'llm', 'large language model', 'langchain', 'llamaindex',
            'haystack', 'semantic kernel', 'openai api', 'anthropic', 'cohere', 'computer vision',
            'cv', 'opencv', 'skimage', 'pillow', 'yolo', 'detectron',
            'mediapipe', 'tesseract', 'ocr', 'image processing', 'segmentation', 'object detection',
            'mlops', 'mlflow', 'kubeflow', 'tfx', 'weights & biases', 'wandb',
            'comet ml', 'neptune.ai', 'dvc', 'pachyderm', 'lakefs', 'bentoml',
            'seldon', 'kserve', 'triton inference server', 'sagemaker', 'vertex ai', 'azure ml',
            'databricks', 'snowflake ml', 'h2o.ai', 'datasns',

            # --- Big Data ---
            'big data', 'hadoop', 'hdfs', 'mapreduce', 'spark', 'apache spark',
            'pyspark', 'spark sql', 'flink', 'apache flink', 'storm', 'samza',
            'beam', 'apache beam', 'tez', 'data warehouse', 'data lake', 'lakehouse',
            'snowflake', 'databricks', 'bigquery', 'redshift', 'synapse analytics', 'hive',
            'impala', 'presto', 'trino', 'drill', 'athena', 'glue',
            'emr', 'etl', 'elt', 'data pipeline', 'airflow', 'apache airflow',
            'prefect', 'dagster', 'luigi', 'oozie', 'nifi', 'apache nifi',
            'streamsets', 'talend', 'informatica', 'matillion', 'fivetran', 'airbyte',
            'dbt', 'data build tool', 'sqoop', 'flume', 'kafka', 'apache kafka',
            'kafka streams', 'confluent', 'pulsar', 'apache pulsar', 'rabbitmq', 'kinesis',
            'pub/sub', 'eventhub', 'spark streaming',

            # --- Databases ---
            'sql', 'mysql', 'postgresql', 'postgres', 'oracle database', 'oracle db',
            'sql server', 'mssql', 't-sql', 'pl/pgsql', 'mariadb', 'sqlite',
            'db2', 'sybase', 'informix', 'firebird', 'cockroachdb', 'tidb',
            'yugabytesql', 'google spanner', 'aurora', 'mongodb', 'couchdb', 'couchbase',
            'raven db', 'marklogic', 'firebase', 'firestore', 'dynamodb', 'cosmos db',
            'documentdb', 'redis', 'memcached', 'etcd', 'riak', 'aerospike',
            'amazon dynamodb', 'azure cosmos db', 'cassandra', 'hbase', 'scylladb', 'accumulo',
            'bigtable', 'neo4j', 'arangodb', 'orientdb', 'janusgraph', 'tigergraph',
            'amazon neptune', 'azure cosmos db gremlin', 'pinecone', 'milvus', 'weaviate', 'qdrant',
            'chroma', 'chromadb', 'faiss', 'elasticsearch vector', 'elasticsearch', 'elastic stack',
            'elk', 'logstash', 'kibana', 'solr', 'lucene', 'algolia',
            'meilisearch', 'typesense', 'opensearch', 'influxdb', 'timescaledb', 'prometheus',
            'graphite', 'opentsdb', 'victoriametrics', 'questdb',

            # --- Cloud & DevOps ---
            'aws', 'amazon web services', 'ec2', 's3', 'lambda', 'rds',
            'dynamodb', 'vpc', 'iam', 'cloudwatch', 'azure', 'microsoft azure',
            'azure vm', 'azure functions', 'azure sql', 'azure ad', 'entra id', 'gcp',
            'google cloud', 'gce', 'gke', 'cloud run', 'cloud functions', 'app engine',
            'oracle cloud', 'oci', 'ibm cloud', 'alibaba cloud', 'digitalocean', 'linode',
            'vultr', 'hetzner', 'heroku', 'render', 'fly.io', 'railway',
            'netlify', 'vercel', 'cloudflare', 'cloudflare workers', 'docker', 'docker compose',
            'docker swarm', 'podman', 'lxc', 'containerd', 'rkt', 'kubernetes',
            'k8s', 'kubectl', 'helm', 'kustomize', 'openshift', 'okd',
            'rancher', 'tanzu', 'eks', 'aks', 'gke', 'minikube',
            'kind', 'k3s', 'istio', 'linkerd', 'consul', 'envoy',
            'traefik', 'nginx', 'apache httpd', 'haproxy', 'kong', 'api gateway',
            'calico', 'cilium', 'coredns', 'coredns', 'terraform', 'opentofu',
            'terragrunt', 'ansible', 'puppet', 'chef', 'saltstack', 'cloudformation',
            'cdk', 'aws cdk', 'azure bicep', 'arm templates', 'pulumi', 'crossplane',
            'vagrant', 'packer', 'jenkins', 'gitlab ci', 'github actions', 'azure devops',
            'ado', 'circleci', 'travis ci', 'bamboo', 'teamcity', 'bitbucket pipelines',
            'drone', 'argo cd', 'flux', 'spinnaker', 'tekton', 'prometheus',
            'grafana', 'alertmanager', 'thanos', 'cortex', 'loki', 'jaeger',
            'zipkin', 'tempo', 'datadog', 'new relic', 'dynatrace', 'splunk',
            'sumo logic', 'appdynamics', 'zabbix', 'nagios', 'icinga', 'sentry',
            'ig', 'elk stack',

            # --- Cybersecurity ---
            'cybersecurity', 'information security', 'infosec', 'network security', 'application security', 'appsec',
            'cloud security', 'devsecops', 'endpoint security', 'zero trust', 'encryption', 'cryptography',
            'pki', 'ssl', 'tls', 'ssh', 'vpn', 'ipsec',
            'firewall', 'waf', 'ids', 'ips', 'dlp', 'siem',
            'soar', 'edr', 'xdr', 'iam', 'pam', 'sso',
            'mfa', '2fa', 'oauth', 'oauth2', 'openid connect', 'oidc',
            'saml', 'ldap', 'active directory', 'ad', 'kerberos', 'okta',
            'auth0', 'ping identity', 'onelogin', 'penetration testing', 'pentesting', 'ethical hacking',
            'red teaming', 'vulnerability assessment', 'vulnerability management', 'bug bounty', 'ctf', 'metasploit',
            'burp suite', 'owasp zap', 'nmap', 'wireshark', 'nessus', 'qualys',
            'openvas', 'hashcat', 'john the ripper', 'aircrack-ng', 'sqlmap', 'cobalt strike',
            'gdpr', 'hipaa', 'pci dss', 'iso 27001', 'soc 2', 'nist',
            'cis controls', 'fisma', 'fedramp',

            # --- Business & Other ---
            'blockchain', 'dlt', 'cryptocurrency', 'bitcoin', 'ethereum', 'solana',
            'smart contracts', 'web3', 'salesforce', 'crm', 'sap', 'erp',
            'oracle ebs', 'workday', 'shopify', 'magento', 'e-commerce', 'agile',
            'scrum', 'kanban', 'safe', 'devops', 'tdd', 'bdd',
            'microservices', 'serverless', 'jira', 'confluence', 'microsoft office', 'excel',
            'power bi', 'tableau', 'networking', 'tcp/ip', 'dns', 'cisco',
            'ccna', 'iot', 'mqtt', 'arduino', 'raspberry pi',
        ]

        # Sort by length descending to ensure longer phrases match first (e.g. "node.js" vs "node")
        SKILLS.sort(key=len, reverse=True)

        # Create mapping for canonical casing
        extract_skills_from_text.mapping = {s.lower(): s for s in SKILLS}

        # Compile optimized regex with lookaround boundaries
        # (?<!\w) checks that start is not preceded by a word char
        # (?!\w) checks that end is not followed by a word char
        # This correctly handles "c++" (ends with non-word) and "node" (ends with word)

        import re
        # Re-import re inside to be safe with cell scope if needed, though global import is standard
        pattern_str = r'(?<!\w)(?:' + '|'.join(re.escape(s.lower()) for s in SKILLS) + r')(?!\w)'
        extract_skills_from_text.pattern = re.compile(pattern_str, re.IGNORECASE)

    # Find all matches (returns list of matched substrings)
    matches = extract_skills_from_text.pattern.findall(text.lower())

    # Map back to canonical case and deduplicate
    found_skills = {extract_skills_from_text.mapping.get(m, m.title()) for m in matches}

    return list(found_skills)


In [5]:
# Step 5: Define IT Relevance Check and Job Scraping Functions

def is_it_related_job(title, company, description, industries, location=None):
    """

    # Sri Lankan Locations for Validation
    valid_locations = [
        'sri lanka',
        'colombo', 'kandy', 'galle', 'jaffna', 'negombo', 'gampaha', 'matara',
        'trincomalee', 'batticaloa', 'kurunegala', 'ratnapura', 'anuradhapura',
        'badulla', 'nuwara eliya', 'kalutara', 'matale', 'puttalam', 'kegalle',
        'mannar', 'vavuniya', 'mullaitivu', 'kilinochchi', 'polonnaruwa',
        'monaragala', 'hambantota', 'ampara',
        'western province', 'central province', 'southern province',
        'northern province', 'eastern province', 'north western province',
        'north central province', 'uva province', 'sabaragamuwa province'
    ]

    # Check location strictly
    if location:
        loc_lower = location.lower()
        if not any(valid_loc in loc_lower for valid_loc in valid_locations):
            # Special case: 'Remote' jobs might not have location, but if location is provided and NOT SL, reject.
            # However, if location is 'Remote' it might still be valid if the search context is SL.
            # But user asked for 'only location in Sri Lanka'.
            # So if location string is present, it MUST contain one of the valid terms.
            return False

    Check if a job is IT-related based on title, description, and industries
    Returns True if job is relevant to IT industry
    """
    if not title:
        return False

    # Comprehensive IT-related keywords in job titles
    it_title_keywords = [

    # =====================================================
    # --- Core Development & Engineering ---
    # =====================================================
    'software engineer', 'software developer', 'application developer',
    'programmer', 'coder', 'software architect',
    'systems engineer', 'systems developer',
    'frontend engineer', 'frontend developer',
    'backend engineer', 'backend developer',
    'fullstack engineer', 'full stack developer',
    'web developer', 'web engineer',
    'mobile developer', 'mobile engineer',
    'app developer', 'application engineer',
    'desktop developer',
    'ios developer', 'android developer',
    'kotlin developer', 'swift developer',
    'game developer', 'unity developer', 'unreal developer',
    'embedded developer', 'embedded systems engineer',
    'firmware engineer', 'hardware engineer',

    # =====================================================
    # --- Seniority Variants ---
    # =====================================================
    'junior developer', 'senior developer',
    'junior engineer', 'senior engineer',
    'lead developer', 'lead engineer',
    'principal engineer', 'staff engineer',
    'technical lead', 'team lead',
    'engineering manager', 'senior engineering manager',
    'director of engineering', 'vp engineering',
    'head of engineering',
    'chief technology officer', 'cto',
    'chief information officer', 'cio',
    'chief digital officer',

    # =====================================================
    # --- DevOps / Cloud / Infrastructure ---
    # =====================================================
    'devops', 'devops engineer',
    'site reliability engineer', 'sre',
    'cloud engineer', 'cloud architect',
    'cloud consultant',
    'platform engineer', 'platform architect',
    'infrastructure engineer',
    'build engineer', 'release engineer',
    'configuration engineer',
    'kubernetes engineer', 'docker engineer',
    'terraform engineer',
    'linux engineer',
    'cloud operations engineer',
    'cloud security engineer',

    # =====================================================
    # --- QA / Testing ---
    # =====================================================
    'qa engineer', 'quality assurance engineer',
    'test engineer', 'automation engineer',
    'sdet', 'tester',
    'manual tester', 'performance tester',
    'qa analyst', 'test automation engineer',

    # =====================================================
    # --- Data & Analytics ---
    # =====================================================
    'data scientist', 'senior data scientist',
    'data analyst', 'business intelligence analyst',
    'data engineer', 'analytics engineer',
    'data architect',
    'machine learning engineer', 'ml engineer',
    'ai engineer', 'ai architect',
    'deep learning engineer',
    'bi developer',
    'etl developer',
    'data warehouse engineer',
    'big data engineer',
    'analytics manager',
    'statistician',

    # =====================================================
    # --- AI / GenAI / Advanced AI ---
    # =====================================================
    'ai researcher', 'research scientist',
    'nlp engineer', 'natural language processing engineer',
    'computer vision engineer',
    'generative ai engineer',
    'llm engineer', 'large language model engineer',
    'prompt engineer',
    'mlops engineer',
    'ai solutions architect',

    # =====================================================
    # --- Security ---
    # =====================================================
    'security engineer', 'security architect',
    'cybersecurity engineer',
    'cyber security analyst',
    'infosec engineer',
    'information security analyst',
    'security analyst',
    'network security engineer',
    'cloud security architect',
    'application security engineer',
    'penetration tester',
    'ethical hacker',
    'soc analyst',
    'incident responder',
    'security consultant',
    'grc analyst',

    # =====================================================
    # --- Networking / Systems / Database ---
    # =====================================================
    'network engineer',
    'network administrator',
    'system administrator',
    'sysadmin',
    'systems engineer',
    'it administrator',
    'database administrator',
    'dba',
    'database engineer',
    'storage engineer',

    # =====================================================
    # --- Product / Project / Agile ---
    # =====================================================
    'product manager',
    'technical product manager',
    'project manager',
    'it project manager',
    'program manager',
    'scrum master',
    'agile coach',
    'delivery manager',
    'product owner',
    'release manager',

    # =====================================================
    # --- Design / UX ---
    # =====================================================
    'ui designer',
    'ux designer',
    'ui ux designer',
    'product designer',
    'interaction designer',
    'visual designer',
    'ux researcher',
    'design system engineer',

    # =====================================================
    # --- Enterprise / Consulting ---
    # =====================================================
    'solutions architect',
    'enterprise architect',
    'technical consultant',
    'technology consultant',
    'implementation engineer',
    'integration specialist',
    'sap consultant',
    'salesforce developer',
    'oracle consultant',

    # =====================================================
    # --- IT Support / Operations ---
    # =====================================================
    'it support',
    'technical support engineer',
    'help desk',
    'service desk analyst',
    'it specialist',
    'it technician',
    'desktop support engineer',
    'it operations engineer',
    'support engineer',

    # =====================================================
    # --- Emerging / Niche Tech ---
    # =====================================================
    'blockchain developer',
    'web3 developer',
    'smart contract developer',
    'robotics engineer',
    'iot engineer',
    'quantum computing engineer',
    'ar developer',
    'vr developer',
    'mixed reality developer',
    'metaverse developer',
    'rpa developer',
    'automation architect',
]


    # Comprehensive IT-related industries
    it_industries_keywords = [

        # ==============================
        # --- Core Technology ---
        # ==============================
        'software development',
        'information technology',
        'computer software',
        'internet',
        'technology',
        'it services',
        'it consulting',
        'computer networking',
        'computer hardware',
        'telecommunications',
        'semiconductors',
        'electronics',
        'computer engineering',

        # ==============================
        # --- Cloud / AI / Data ---
        # ==============================
        'cloud computing',
        'artificial intelligence',
        'machine learning',
        'data analytics',
        'big data',
        'data science',
        'cybersecurity',
        'blockchain',
        'web3',
        'robotics',
        'automation',
        'saas',
        'paas',
        'iaas',

        # ==============================
        # --- Digital Services ---
        # ==============================
        'e-learning',
        'edtech',
        'e-commerce',
        'digital marketing',
        'online media',
        'information services',
        'gaming',
        'mobile applications',
        'digital transformation',

        # ==============================
        # --- Financial & Enterprise ---
        # ==============================
        'financial services',
        'fintech',
        'banking',
        'insurance',
        'investment management',
        'payments',
        'enterprise software',
        'business consulting',
        'management consulting',
        'outsourcing',
        'staffing',
        'erp',
        'crm',

        # ==============================
        # --- Tech-Heavy Industries ---
        # ==============================
        'healthtech',
        'medtech',
        'biotechnology',
        'automotive technology',
        'aerospace',
        'defense technology',
        'logistics technology',
        'supply chain technology',
        'retail technology',
        'proptech',
        'agritech',
        'energy technology',
        'smart manufacturing',
    ]


    # Check title
    title_lower = title.lower()
    if any(keyword in title_lower for keyword in it_title_keywords):
        return True

    # Check industries
    if industries:
        industries_lower = industries.lower()
        if any(keyword in industries_lower for keyword in it_industries_keywords):
            return True

    # Check description (if available)
    if description:
        desc_lower = description.lower()
        # Look for IT-related terms in description
        it_desc_keywords = ['software', 'programming', 'coding', 'development', 'technical', 'technology']
        match_count = sum(1 for keyword in it_desc_keywords if keyword in desc_lower)
        if match_count >= 2:  # At least 2 IT keywords in description
            return True

    return False

def get_job_details(job_id):
    """Get detailed job information from LinkedIn API"""
    url = f"https://www.linkedin.com/jobs-guest/jobs/api/jobPosting/{job_id}"

    try:
        response = safe_get(url)
        if not response:
            return {}

        soup = BeautifulSoup(response.text, 'html.parser')
        details = {}

        # Extract description
        desc_elem = soup.find("div", {"class": "show-more-less-html__markup"})
        if desc_elem:
            description = desc_elem.get_text(separator=" ", strip=True)
            details["description"] = description
            # Extract skills from description
            skills = extract_skills_from_text(description)
            details["required_skills"] = ', '.join(skills) if skills else None
        else:
            details["description"] = None
            details["required_skills"] = None

        # Extract job criteria (stored as single field and individual fields)
        criteria_list = soup.find("ul", {"class": "description__job-criteria-list"})
        criteria_text_list = []

        if criteria_list:
            for item in criteria_list.find_all("li"):
                header = item.find("h3", {"class": "description__job-criteria-subheader"})
                value = item.find("span", {"class": "description__job-criteria-text"})

                if header and value:
                    header_text = header.text.strip()
                    value_text = value.text.strip()
                    criteria_text_list.append(f"{header_text}: {value_text}")

                    if "Seniority level" in header_text:
                        details["experience_level"] = value_text
                    elif "Employment type" in header_text:
                        details["employment_type"] = value_text
                    elif "Job function" in header_text:
                        details["job_function"] = value_text
                    elif "Industries" in header_text:
                        details["industries"] = value_text

        # Store combined job criteria
        details["job_criteria"] = " | ".join(criteria_text_list) if criteria_text_list else None

        # Extract number of applicants
        try:
            num_applicants_elem = soup.find("span", {"class": "num-applicants__caption"})
            if not num_applicants_elem:
                num_applicants_elem = soup.find("figcaption", {"class": "num-applicants__caption"})

            if num_applicants_elem:
                details["num_applicants"] = num_applicants_elem.get_text(strip=True)
            else:
                details["num_applicants"] = None
        except:
            details["num_applicants"] = None

        # Ensure all required fields exist with None if not found
        for field in ["experience_level", "employment_type", "job_function", "industries"]:
            if field not in details:
                details[field] = None

        return details

    except Exception as e:
        logger.error(f"Error fetching details for job {job_id}: {e}")
        return {}

def extract_job_card_data(card, search_keyword=""):
    """Extract data from a job card HTML element"""
    try:
        # Get job link and ID
        job_link = card.find("a", {"class": "base-card__full-link"})
        if not job_link:
            return None

        job_url = job_link.get('href', '').split('?')[0]
        job_id = job_url.split('-')[-1] if job_url else None

        if not job_id:
            return None

        # Extract basic info
        title_elem = card.find("h3", {"class": "base-search-card__title"})
        company_elem = card.find("h4", {"class": "base-search-card__subtitle"})
        location_elem = card.find("span", {"class": "job-search-card__location"})
        time_elem = card.find("time")

        job_data = {
            "job_id": job_id,
            "title": title_elem.text.strip() if title_elem else None,
            "company": company_elem.text.strip() if company_elem else None,
            "location": location_elem.text.strip() if location_elem else None,
            "posted_date": time_elem.get("datetime") if time_elem else None,
            "job_url": job_url,
            "search_keyword": search_keyword,
            "scraped_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        }

        # Get detailed info (with delay to avoid rate limiting)
        time.sleep(random.uniform(1, 2))
        details = get_job_details(job_id)
        job_data.update(details)

        return job_data

    except Exception as e:
        logger.error(f"Error extracting job card data: {e}")
        return None

print("Scraping functions defined successfully")

Scraping functions defined successfully


In [6]:
# Step 6: Main Scraping Function for Single Keyword
def scrape_jobs_by_keyword(keyword, seen_ids=None):
    """
    Scrape LinkedIn IT jobs for a single keyword in Sri Lanka
    Goes through ALL available pages for the keyword
    """
    if seen_ids is None:
        seen_ids = set()

    jobs_data = []
    location = CONFIG['location'].replace(' ', '%20')
    keyword_encoded = keyword.replace(' ', '%20')

    # Simple URL - just keyword and location (no filters)
    base_url = (
        f"https://www.linkedin.com/jobs-guest/jobs/api/seeMoreJobPostings/search?"
        f"keywords={keyword_encoded}&"
        f"location={location}&"
        f"start={{}}"
    )

    max_pages = CONFIG['max_pages_per_keyword']

    logger.info(f"Scraping keyword: '{keyword}'")

    for page in range(max_pages):
        try:
            url = base_url.format(page * 25)
            response = safe_get(url)

            if not response:
                logger.warning(f"  Failed to get page {page} for '{keyword}'")
                break

            # Check for no results message
            if "No matching jobs found" in response.text or "No exact matches found" in response.text:
                if page == 0:
                    logger.info(f"  No jobs found for '{keyword}'. Skipping to next.")
                else:
                    logger.info(f"  No more jobs for '{keyword}'.")
                break

            soup = BeautifulSoup(response.text, 'html.parser')
            job_cards = soup.find_all("div", {"class": "base-card"})

            if not job_cards:
                logger.info(f"  No more jobs found at page {page} for '{keyword}'")
                break

            page_new_jobs = 0
            for card in job_cards:
                job_data = extract_job_card_data(card, search_keyword=keyword)

                if job_data and job_data['job_id'] not in seen_ids:
                    # Check if IT-related
                    if is_it_related_job(
                        job_data.get('title'),
                        job_data.get('company'),
                        job_data.get('description'),
                        job_data.get('industries'),
                        job_data.get('location')
                    ):
                        seen_ids.add(job_data['job_id'])
                        jobs_data.append(job_data)
                        page_new_jobs += 1

            if page_new_jobs > 0:
                logger.info(f"  Page {page}: Found {page_new_jobs} new IT jobs")

            # Delay between pages
            time.sleep(random.uniform(*CONFIG['page_delay']))

        except Exception as e:
            logger.error(f"  Error on page {page} for '{keyword}': {e}")
            continue

    logger.info(f"Keyword '{keyword}' complete: {len(jobs_data)} new jobs")
    return jobs_data

print("Keyword-based scraping function ready")

Keyword-based scraping function ready


## Execute Keyword-by-Keyword Scraping

**Run this cell to scrape IT jobs using each keyword separately.**

This will:
- Search each keyword one by one (100+ keywords)
- Scrape ALL available pages for each keyword
- Save/append results to CSV after each keyword
- Deduplicate across all keywords by job_id
- Accept nulls for missing data fields
- Resume-friendly: Can stop and restart without losing data

**Estimated time:** 2-4 hours (depending on results per keyword)

**Note**: Progress is saved after each keyword, so you can safely stop and resume.

In [7]:
# Step 7: Execute Keyword-by-Keyword Scraping with Progressive Saving
import os

print("="*60)
print("KEYWORD-BY-KEYWORD SCRAPING - SRI LANKAN IT JOBS")
print("="*60)
print(f"Total keywords: {len(CONFIG['search_keywords'])}")
print(f"Max pages per keyword: {CONFIG['max_pages_per_keyword']}")
print("="*60 + "\n")

# Setup output
output_dir = Path(CONFIG['output_dir'])
output_dir.mkdir(exist_ok=True)
csv_filepath = output_dir / CONFIG['output_filename']

# Track progress
seen_ids = set()
total_jobs_collected = 0
keyword_stats = []

# Load existing data if CSV exists (for resuming)
if csv_filepath.exists():
    print(f"Found existing CSV: {csv_filepath}")
    existing_df = pd.read_csv(csv_filepath)
    seen_ids = set(existing_df['job_id'].astype(str).tolist())
    total_jobs_collected = len(existing_df)
    print(f"Loaded {total_jobs_collected} existing jobs\n")
else:
    print(f"Starting fresh: {csv_filepath}\n")

# Iterate through each keyword
for idx, keyword in enumerate(CONFIG['search_keywords'], 1):
    print(f"\n[{idx}/{len(CONFIG['search_keywords'])}] Searching: '{keyword}'")

    # Scrape jobs for this keyword
    new_jobs = scrape_jobs_by_keyword(keyword, seen_ids)

    if new_jobs:
        # Convert to DataFrame
        df_new = pd.DataFrame(new_jobs)

        # Append to CSV (create if doesn't exist)
        if not csv_filepath.exists():
            df_new.to_csv(csv_filepath, index=False, encoding='utf-8-sig', mode='w')
            print(f"Created CSV with {len(df_new)} jobs")
        else:
            df_new.to_csv(csv_filepath, index=False, encoding='utf-8-sig', mode='a', header=False)
            print(f"Appended {len(df_new)} jobs to CSV")

        total_jobs_collected += len(df_new)
        keyword_stats.append({
            'keyword': keyword,
            'jobs_found': len(df_new),
            'total_so_far': total_jobs_collected
        })
    else:
        keyword_stats.append({
            'keyword': keyword,
            'jobs_found': 0,
            'total_so_far': total_jobs_collected
        })

    print(f"Running total: {total_jobs_collected} unique jobs")

    # Delay between keywords
    if idx < len(CONFIG['search_keywords']):
        delay = random.uniform(*CONFIG['keyword_delay'])
        print(f"Waiting {delay:.1f}s before next keyword...")
        time.sleep(delay)

print("\n" + "="*60)
print("SCRAPING COMPLETE - PERFORMING FINAL DEDUPLICATION")
print("="*60)

# Load final dataset and perform thorough deduplication
if csv_filepath.exists():
    df_full = pd.read_csv(csv_filepath)
    initial_count = len(df_full)

    print(f"\nInitial records: {initial_count}")

    # Deduplicate by job_id (keep first occurrence)
    df_full = df_full.drop_duplicates(subset=['job_id'], keep='first')
    final_count = len(df_full)
    duplicates_removed = initial_count - final_count

    print(f"After deduplication: {final_count}")
    print(f"Duplicates removed: {duplicates_removed}")

    # Save the final deduplicated dataset
    final_csv_filepath = output_dir / "linkedin_sri_lanka_IT_jobs_final.csv"
    df_full.to_csv(final_csv_filepath, index=False, encoding='utf-8-sig')
    print(f"\nFinal CSV saved: {final_csv_filepath}")

    # Remove the progressive CSV file (keep only final)
    if csv_filepath != final_csv_filepath and csv_filepath.exists():
        try:
            os.remove(csv_filepath)
            print(f"Removed intermediate file: {csv_filepath}")
        except Exception as e:
            print(f"Could not remove intermediate file: {e}")

    print(f"\nSuccessfully scraped {final_count} unique IT jobs")
    print(f"Data shape: {df_full.shape}")

    # Show top keywords
    print(f"\nTop 10 Keywords by Jobs Found:")
    df_stats = pd.DataFrame(keyword_stats)
    top_keywords = df_stats[df_stats['jobs_found'] > 0].nlargest(10, 'jobs_found')
    for _, row in top_keywords.iterrows():
        print(f"  {row['keyword']}: {row['jobs_found']} jobs")

else:
    print("\nNo jobs scraped. Check errors above.")
    df_full = pd.DataFrame()
    final_csv_filepath = None

KEYWORD-BY-KEYWORD SCRAPING - SRI LANKAN IT JOBS
Total keywords: 205
Max pages per keyword: 40

Starting fresh: data/linkedin_sri_lanka_IT_jobs_progressive.csv


[1/205] Searching: 'software engineer'
Created CSV with 86 jobs
Running total: 86 unique jobs
Waiting 10.2s before next keyword...

[2/205] Searching: 'software developer'
Appended 37 jobs to CSV
Running total: 123 unique jobs
Waiting 12.1s before next keyword...

[3/205] Searching: 'software development engineer'
Appended 27 jobs to CSV
Running total: 150 unique jobs
Waiting 14.3s before next keyword...

[4/205] Searching: 'junior software engineer'
Appended 24 jobs to CSV
Running total: 174 unique jobs
Waiting 14.6s before next keyword...

[5/205] Searching: 'senior software engineer'
Appended 9 jobs to CSV
Running total: 183 unique jobs
Waiting 12.2s before next keyword...

[6/205] Searching: 'principal software engineer'
Appended 5 jobs to CSV
Running total: 188 unique jobs
Waiting 11.6s before next keyword...

[7/205] Sea

## Generate Final Files

Final deduplication and file generation:

After scraping completes, the system will:
1. Load all scraped data from progressive CSV
2. Remove duplicates by job_id (keeping first occurrence)
3. Save single final CSV: linkedin_sri_lanka_IT_jobs_final.csv
4. Delete intermediate progressive file
5. Generate Excel file with timestamp
6. Create comprehensive summary report

Result: Only 3 files in the data/ folder:
- 1 CSV file (deduplicated)
- 1 Excel file
- 1 Summary report

In [8]:
# Step 8: Generate Final Excel and Summary Report
if not df_full.empty and final_csv_filepath:
    # Create output directory
    output_dir = Path(CONFIG['output_dir'])
    output_dir.mkdir(exist_ok=True)

    # Generate timestamp for final files
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    excel_filename = output_dir / f"linkedin_sri_lanka_IT_jobs_{timestamp}.xlsx"

    # Save to Excel
    try:
        df_full.to_excel(excel_filename, index=False, engine='openpyxl')
        print(f"Excel file created: {excel_filename}")
    except Exception as e:
        print(f"Excel save failed: {e}")

    # Save comprehensive summary statistics
    summary_filename = output_dir / f"scraping_summary_{timestamp}.txt"
    with open(summary_filename, 'w', encoding='utf-8') as f:
        f.write("="*60 + "\n")
        f.write("LINKEDIN IT JOB SCRAPING SUMMARY\n")
        f.write("Keyword-by-Keyword Strategy\n")
        f.write("="*60 + "\n\n")
        f.write(f"Scrape Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Total IT Jobs Scraped: {len(df_full)}\n")
        f.write(f"Total Keywords Searched: {len(CONFIG['search_keywords'])}\n")
        f.write(f"Location: {CONFIG['location']}\n")
        f.write(f"Strategy: Individual keyword search with full pagination\n\n")
        f.write("="*60 + "\n")
        f.write("TOP 20 KEYWORDS BY JOBS FOUND\n")
        f.write("="*60 + "\n\n")

        if 'search_keyword' in df_full.columns:
            keyword_counts = df_full['search_keyword'].value_counts().head(20)
            for kw, count in keyword_counts.items():
                f.write(f"{kw}: {count} jobs\n")

        f.write("\n" + "="*60 + "\n")
        f.write("DATA COMPLETENESS\n")
        f.write("="*60 + "\n\n")

        for col in df_full.columns:
            missing = df_full[col].isna().sum()
            coverage = ((len(df_full) - missing) / len(df_full)) * 100
            f.write(f"{col}: {coverage:.1f}% ({len(df_full) - missing}/{len(df_full)})\n")

        f.write("\n" + "="*60 + "\n")
        f.write("TOP 10 COMPANIES\n")
        f.write("="*60 + "\n\n")
        if 'company' in df_full.columns:
            top_companies = df_full['company'].value_counts().head(10)
            for company, count in top_companies.items():
                f.write(f"{company}: {count} jobs\n")

        f.write("\n" + "="*60 + "\n")
        f.write("TOP 10 JOB TITLES\n")
        f.write("="*60 + "\n\n")
        if 'title' in df_full.columns:
            top_titles = df_full['title'].value_counts().head(10)
            for title, count in top_titles.items():
                f.write(f"{title}: {count} postings\n")

    print(f"Summary report: {summary_filename}")

    print("\n" + "="*60)
    print("FINAL STATISTICS")
    print("="*60)
    print(f"Total Jobs: {len(df_full)}")
    print(f"Unique Companies: {df_full['company'].nunique() if 'company' in df_full.columns else 'N/A'}")
    print(f"Unique Job Titles: {df_full['title'].nunique() if 'title' in df_full.columns else 'N/A'}")
    print(f"Keywords Used: {len(CONFIG['search_keywords'])}")
    print(f"Keywords with Results: {df_full['search_keyword'].nunique() if 'search_keyword' in df_full.columns else 'N/A'}")
    if 'posted_date' in df_full.columns and df_full['posted_date'].notna().any():
        print(f"Date Range: {df_full['posted_date'].min()} to {df_full['posted_date'].max()}")
    print(f"Data completeness: {df_full.notna().sum().sum() / (len(df_full) * len(df_full.columns)) * 100:.1f}%")
    print("="*60)

    print("\nFILES GENERATED")
    print("="*60)
    print(f"1. CSV: {final_csv_filepath}")
    print(f"2. Excel: {excel_filename}")
    print(f"3. Summary: {summary_filename}")
    print("="*60)

else:
    print("No data to process. Please run scraping first.")

Excel save failed: Service Desk Role Summary The Service Desk Lead is responsible for overseeing day-to-day Service Desk operations, ensuring highquality incident and request management, adherence to SLAs, and effective team performance. The role acts as a bridge between analysts, management, and stakeholders, driving operational excellence, continuous improvement, and customer satisfaction. Key Responsibilities Operational Management • Lead and manage Service Desk analysts across shifts to ensure uninterrupted support coverage. • Monitor and manage Incidents, Service Requests, and Tickets through ITSM tools. • Ensure SLA, KPI, and OLA adherence for all supported services. • Act as an escalation point for critical incidents and major issues. • Coordinate with resolver groups (Network, Security, Application, EUC teams). People & Team Management • Allocate work, manage shift rosters, and ensure optimal resource utilization. • Coach, mentor, and guide Service Desk analysts for performanc

In [9]:
# Step 9: Exploratory Data Analysis
if not df_full.empty:
    print("="*60)
    print("EXPLORATORY DATA ANALYSIS")
    print("="*60)

    print("\n[Top 15 Keywords by Jobs Found]")
    if 'search_keyword' in df_full.columns:
        print(df_full['search_keyword'].value_counts().head(15))

    print("\n[Top 10 Companies by Job Postings]")
    if 'company' in df_full.columns:
        print(df_full['company'].value_counts().head(10))

    print("\n[Top 10 Job Titles]")
    if 'title' in df_full.columns:
        print(df_full['title'].value_counts().head(10))

    print("\n[Location Distribution]")
    if 'location' in df_full.columns:
        print(df_full['location'].value_counts().head(10))

    print("\n[Experience Level Distribution]")
    if 'experience_level' in df_full.columns:
        exp_counts = df_full['experience_level'].value_counts()
        if not exp_counts.empty:
            print(exp_counts)

    print("\n[Employment Type Distribution]")
    if 'employment_type' in df_full.columns:
        emp_counts = df_full['employment_type'].value_counts()
        if not emp_counts.empty:
            print(emp_counts)

    print("\n[Most In-Demand Skills]")
    if 'required_skills' in df_full.columns:
        all_skills = []
        for skills_str in df_full['required_skills'].dropna():
            if skills_str:
                skills_list = [s.strip() for s in skills_str.split(',')]
                all_skills.extend(skills_list)

        if all_skills:
            from collections import Counter
            skill_counts = Counter(all_skills)
            print("\nTop 20 Most In-Demand IT Skills:")
            for skill, count in skill_counts.most_common(20):
                percentage = (count / len(df_full)) * 100
                print(f"  {skill}: {count} jobs ({percentage:.1f}%)")

    print("\n[Data Completeness by Column]")
    completeness = ((df_full.notna().sum() / len(df_full)) * 100).sort_values(ascending=False)
    for col, pct in completeness.items():
        print(f"  {col}: {pct:.1f}%")

    print("\n[Keyword Effectiveness]")
    if 'search_keyword' in df_full.columns:
        print(f"Total keywords searched: {len(CONFIG['search_keywords'])}")
        print(f"Keywords with results: {df_full['search_keyword'].nunique()}")
        print(f"Effectiveness rate: {(df_full['search_keyword'].nunique() / len(CONFIG['search_keywords']) * 100):.1f}%")

    print("\n" + "="*60)
    print("Data exploration complete")
    print("="*60)
else:
    print("No data available for exploration")

EXPLORATORY DATA ANALYSIS

[Top 15 Keywords by Jobs Found]
search_keyword
team lead                        111
software engineer                 86
technical lead                    45
software developer                37
software development engineer     27
junior software engineer          24
data analyst                      22
product manager                   22
data scientist                    18
devops engineer                   16
project manager                   16
azure engineer                    15
engineering manager               14
application developer             11
c# developer                      11
Name: count, dtype: int64

[Top 10 Companies by Job Postings]
company
LSEG                       57
HCLTech Sri Lanka          39
Virtusa                    32
EY                         24
IGT1                       23
Dialog Axiata PLC          18
Smart Quest Consultancy    14
Axiata Digital Labs        12
Therighttalent             11
Dijital Team               11
N

## Summary & Next Steps

### Successfully Completed - Keyword-by-Keyword Strategy:
- Individual Keyword Search: Each of 100+ keywords searched separately
- Complete Pagination: All available pages scraped per keyword
- Progressive Saving: Data saved after each keyword (safe from failures)
- Maximum Coverage: Different keywords surface different jobs
- No Filters: Simple search (keyword + location only)
- Smart Deduplication: job_id tracking across all keywords
- Resumable: Can stop and restart without losing progress
- Comprehensive IT Focus: 100+ role types, technologies, specializations
- 500+ Skill Extraction: Modern IT skills from job descriptions
- Null-Friendly: Missing data gracefully handled

### Dataset Features:

**Search Keywords Used (100+ total)**:
- Roles: developer, engineer, analyst, designer, manager, architect
- Specializations: frontend, backend, fullstack, devops, data scientist
- Technologies: python, java, react, kubernetes, aws, azure
- Domains: security, QA, UI/UX, database, cloud, AI/ML

**Data Fields Collected**:
- Core: job_id, title, company, location, posted_date, job_url
- Details: description, experience_level, employment_type, job_function, industries
- Skills: required_skills (500+ IT skills extracted)
- Metadata: search_keyword, job_criteria, num_applicants, scraped_at

### Why This Approach is Better:

| Aspect | Old Approach | New Keyword-by-Keyword |
|--------|--------------|------------------------|
| Coverage | Combined OR keywords | Each keyword searched separately |
| Pagination | Limited by filters | ALL pages per keyword |
| Resilience | Lose all if fails | Progressive save per keyword |
| Resumable | No | Yes - can restart anytime |
| Deduplication | Per filter combo | Across all keywords |
| Flexibility | Fixed filters | Pure keyword-based |
| Max Jobs | ~4000 | 100+ keywords x 1000 jobs = 100,000+ potential |

### Output Files:
1. Final CSV: linkedin_sri_lanka_IT_jobs_final.csv (deduplicated, single file)
2. Excel: linkedin_sri_lanka_IT_jobs_{timestamp}.xlsx
3. Summary Report: scraping_summary_{timestamp}.txt (keyword stats, top findings)

Note: Only ONE final CSV file is saved after complete deduplication. Any intermediate files are automatically removed.

### For Your ML Project:

#### 1. You Now Have Maximum IT Job Coverage
- Every possible IT keyword searched
- All pages scraped for each
- Comprehensive dataset for training

#### 2. Keyword Analysis Possible
- Which keywords yield most jobs?
- Which roles are most in-demand?
- Emerging vs traditional roles

#### 3. Ready for ML Pipeline
```python
# Example preprocessing
- Skill matrix (500+ binary features)
- Job title embeddings (BERT)
- Company clustering
- Keyword-based job categorization
- Time series analysis (by search_keyword)
```

#### 4. Resume-Based Job Matching
```python
# Your ML model can:
1. Extract skills from candidate resume
2. Match against scraped jobs
3. Rank by skill overlap
4. Recommend learning paths for skill gaps
5. Show which keywords/roles match best
```

### Important Notes:
1. Scraping Time: 2-4 hours for all 100+ keywords (with delays)
2. Progressive Save: Safe to stop anytime - resume later
3. Rate Limits: Built-in delays respect LinkedIn's servers
4. Deduplication: Automatic across all keywords
5. Educational Use: For academic projects only

### SDG 8 Alignment:
- Comprehensive Job Market View: All IT roles covered
- Better Job Matching: Keyword-level granularity
- Skill Development: Identifies in-demand skills by role type
- Career Planning: Shows demand across different specializations

Your dataset is now ready for advanced ML analysis!